# Colab-X-Local-Model server
Run every cell top to bottom. The last cell blocks and prints a `Public tunnel URL` line — copy that URL into `ui/index.html`'s `BASE_URL` constant.

To use a different model, change `MODEL_NAME` in the cell below before running it — nothing else needs to change.

In [ ]:
!pip install -q flask==3.0.3 transformers==4.44.2 torch==2.4.1 pyngrok==7.2.0

In [ ]:
# The one place to change the model — pick per experiment.
MODEL_NAME = "gpt2"

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model=MODEL_NAME)

# Sanity check inside the notebook before wiring up Flask.
print(generator("Hello, my name is", max_new_tokens=20, num_return_sequences=1))

In [ ]:
from flask import Flask, jsonify, request

app = Flask(__name__)


@app.post("/generate")
def generate():
    data = request.get_json(silent=True) or {}
    prompt = data.get("prompt", "")
    if not isinstance(prompt, str) or not prompt.strip():
        return jsonify({"error": "prompt is required"}), 400

    max_new_tokens = data.get("max_new_tokens", 200)
    outputs = generator(prompt, max_new_tokens=max_new_tokens, num_return_sequences=1)
    text = outputs[0]["generated_text"]
    if text.startswith(prompt):
        text = text[len(prompt):].lstrip()
    return jsonify({"response": text})

In [ ]:
# Set your ngrok authtoken (free account at ngrok.com). Only needed once per Colab runtime.
from pyngrok import ngrok

ngrok.set_auth_token("<YOUR_NGROK_AUTHTOKEN>")

In [ ]:
# This cell blocks — the tunnel URL is printed above the running server log.
public_url = ngrok.connect(5000)
print(f"Public tunnel URL: {public_url}")
print("Copy this into ui/index.html's BASE_URL constant.")

app.run(host="0.0.0.0", port=5000)